In [9]:
import logging
import os
from tqdm import tqdm
import SimpleITK as sitk
import numpy as np
import sys
from pathlib import Path
from random import randint

log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)

MRI_FOLDER = "data/raw/images/"
ANNOTATION_FOLDER = "data/raw/labels/"
OUTPUT_DIR = "output/extract1"

def setup_logger():
    logger = logging.getLogger(__name__)
    
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
        handler.close()
    
    logger.setLevel(logging.DEBUG)
    
    logger.propagate = False
    log_file = os.path.join(log_dir, 'logs_2dx3.log')
    
    # Add file handler
    file_handler = logging.FileHandler(log_file)
    file_handler.setLevel(logging.DEBUG)
    
    # Add console handler
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO) 
    
    # Create formatter
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    console_handler.setFormatter(formatter)
    
    # Add handlers
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)
    
    return logger

# Initialize logger
logger = setup_logger()

SW_STRIDE = 1
IMG_PADDING = 3 # TODO: confirm unit


logger.info(f"Starting parameter logging")
logger.info(f"SW_STRIDE: {SW_STRIDE}")
logger.info(f"IMG_PADDING: {IMG_PADDING}")
logger.debug("Debug logging is enabled")

2025-07-18 14:01:23,811 - INFO - Starting parameter logging
2025-07-18 14:01:23,813 - INFO - SW_STRIDE: 1
2025-07-18 14:01:23,814 - INFO - IMG_PADDING: 3


In [10]:
def match_files(mri_files, annotation_files):
    """Match MRI files with their corresponding annotation files based on filename."""
    pairs = []
    matched_annotation_files = set()
    
    for mri_file in mri_files:
        # Extract the base filename without path
        mri_basename = os.path.basename(mri_file)
        
        # Look for a matching annotation file
        for anno_file in annotation_files:
            if os.path.basename(anno_file) == mri_basename:
                pairs.append((mri_file, anno_file))
                matched_annotation_files.add(anno_file)
                break

    for anno_file in annotation_files:
        if anno_file not in matched_annotation_files:
            logger.info(f"No MRI file found for annotation file: {anno_file}")
    
    return pairs

In [11]:
# check for incorrect file names of annotation files first
mri_folder = MRI_FOLDER
annotation_folder = ANNOTATION_FOLDER

# Get all files in both folders
mri_files = [os.path.join(mri_folder, f) for f in os.listdir(mri_folder) 
            if f.endswith('.nii.gz')]

annotation_files = [os.path.join(annotation_folder, f) for f in os.listdir(annotation_folder) 
                if f.endswith('.nii.gz')]

# Match MRI files with corresponding annotation files
file_pairs = match_files(mri_files, annotation_files)


logger.info(f"Found {len(file_pairs)} matching pairs out of {len(mri_files)} MRI files and {len(annotation_files)} annotation files")


2025-07-18 14:01:23,855 - INFO - Found 172 matching pairs out of 217 MRI files and 172 annotation files


In [12]:
class DataLoader:
    def __init__(self, mri_path, annotation_path):
        self.mri_path = mri_path
        self.annotation_path = annotation_path

        self.mri_image = None
        self.annotation_image = None
        self.mri_np = None
        self.annotation_np = None
        self.spacing = None
        self.origin = None
        self.size = None
        self.node_labels = None
        self.node_stats = {}
        self.node_masks = {}

        logger.info("DataLoader initialized")

    def load_data(self):
        """Load MRI and annotation data + some checking."""
        logger.info(f"Loading MRI image from {self.mri_path}")
        self.mri_image = sitk.ReadImage(self.mri_path)
        self.mri_np = sitk.GetArrayFromImage(self.mri_image)
        
        logger.info(f"Loading annotation image from {self.annotation_path}")
        self.annotation_image = sitk.ReadImage(self.annotation_path)
        self.annotation_np = sitk.GetArrayFromImage(self.annotation_image)

        # Ensure same coordinate system
        if not self.check_coordinate_match():
            logger.warning("MRI and annotation images might not be in the same coordinate system!")
        
        self.spacing = self.mri_image.GetSpacing()
        logger.info(f"Image spacing: {self.spacing}")

        self.origin = self.mri_image.GetOrigin()
        logger.info(f"Image origin: {self.origin}")

        self.size = self.mri_image.GetSize()
        logger.info(f"Image size: {self.size}")
        
        # Extract node labels
        np_annotation = sitk.GetArrayFromImage(self.annotation_image)
        self.node_labels = np.unique(np_annotation)
        self.node_labels = self.node_labels[self.node_labels > 0]  # Remove background
        
        logger.info(f"Found {len(self.node_labels)} lymph node annotations with labels: {self.node_labels}")
        
        # Create individual masks for each node
        self.create_node_masks()
        
        return self
        
    def check_coordinate_match(self):
        """Helper for the load_data function"""
        """Check if MRI and annotation images have matching coordinate systems."""
        mri_size = self.mri_image.GetSize()
        anno_size = self.annotation_image.GetSize()
        mri_spacing = self.mri_image.GetSpacing()
        anno_spacing = self.annotation_image.GetSpacing()
        mri_origin = self.mri_image.GetOrigin()
        anno_origin = self.annotation_image.GetOrigin()
        
        size_match = mri_size == anno_size
        spacing_match = all(abs(m - a) < 1e-3 for m, a in zip(mri_spacing, anno_spacing))
        origin_match = all(abs(m - a) < 1e-3 for m, a in zip(mri_origin, anno_origin))

        self.num_slides = anno_size[2]
        
        logger.info(f"Size match: {size_match}, Spacing match: {spacing_match}, Origin match: {origin_match}")
        logger.info(f"MRI spacing: {mri_spacing}, Anno spacing: {anno_spacing}")
        logger.info(f"xyz: {anno_size}, num_slides: {self.num_slides}")
        
        return size_match and spacing_match and origin_match
    
    def create_node_masks(self):
        """Helper for the load_data function"""
        """Create binary masks for each lymph node."""
        for label in self.node_labels:
            logger.info(f"Creating mask for node {label}")
            
            # Create binary mask for this node
            node_mask = sitk.Equal(self.annotation_image, int(label))
            self.node_masks[label] = node_mask
            
            # Calculate basic statistics for this node
            np_mri = sitk.GetArrayFromImage(self.mri_image)
            np_mask = sitk.GetArrayFromImage(node_mask)
            node_voxels = np_mri[np_mask > 0]
            
            if len(node_voxels) > 0:
                self.node_stats[label] = {
                    'mean_intensity': np.mean(node_voxels),
                    'std_intensity': np.std(node_voxels),
                    'volume_mm3': np.sum(np_mask) * np.prod(self.spacing),
                    'voxel_count': np.sum(np_mask)
                }
                logger.info(f"  Node {label} stats: {self.node_stats[label]}")
            else:
                logger.warning(f"  Node {label} has no voxels!")

    def get_slices_with_mask(self, node_label):
        """
        Returns a list of slice IDs where the specified node mask exists.
        
        Args:
            node_label: The label of the node to check
            
        Returns:
            List of slice IDs (z-indices) containing the mask
        """
        if node_label not in self.node_masks:
            logger.error(f"Node label {node_label} not found in node masks!")
            return []
        
        # Convert the SimpleITK mask to a numpy array
        mask_array = sitk.GetArrayFromImage(self.node_masks[node_label]) > 0
        
        # Find slices where the mask has at least one True value
        # The first dimension in the numpy array corresponds to the z-axis (slices)
        slices_with_mask = []
        for slice_id in range(mask_array.shape[0]):
            if np.any(mask_array[slice_id]):
                slices_with_mask.append(slice_id)
        
        logger.debug(f"Node {node_label} appears in {len(slices_with_mask)} slices: {slices_with_mask}")
        
        return slices_with_mask


In [13]:
def pad_to_size(img, target_h, target_w):
    h, w = img.shape
    pad_h = (target_h - h) // 2
    pad_w = (target_w - w) // 2
    
    padded = np.zeros((target_h, target_w), dtype=img.dtype)
    padded[pad_h:pad_h+h, pad_w:pad_w+w] = img
    return padded

In [14]:
def get2dx3(label, label_masks, id_list, mri_np, spacing, origin):
    np_label_masks = sitk.GetArrayFromImage(label_masks)
    triplets = []
    list_image_stack_sitk = []
    list_mask_stack_sitk = []

    if len(id_list) == 1:
        logger.info(f"id_list length is 1, no further processing will be done")
        return [], []
    elif len(id_list) == 2:
        randint = randint(0, 1)
        logger.debug(f"randint is {randint}")
        if randint == 0:
            triplets = [((id_list[0]-1), id_list[0], id_list[1])]
        else: 
            triplets = [(id_list[0], id_list[1], (id_list[1]))]

        logger.info(f"id_list length is 2, generated triplet list {triplets}")
    else:
        triplets = [(id_list[i], id_list[i+1], id_list[i+2]) 
                    for i in range(0, len(id_list)-2, SW_STRIDE)]
        logger.info(f"id_list length is 3 or above, generated triplet list {triplets}")
        
    slice_crops = {}

    for slice_id in id_list:
        np_slice_mask = np_label_masks[slice_id]
        rows, cols = np.where(np_slice_mask > 0)

        if len(rows) == 0 or len(cols) == 0:
            continue

        min_row, max_row = np.min(rows), np.max(rows)
        min_col, max_col = np.min(cols), np.max(cols)

        width = max_col - min_col
        height = max_row - min_row

        side = max(width, height)

        centroid_row = (min_row + max_row) // 2
        centroid_col = (min_col + max_col) // 2

        half_side = side // 2

        box_min_row = max(0, centroid_row - half_side - IMG_PADDING)
        box_max_row = min(mri_np.shape[1] - 1, centroid_row + half_side + IMG_PADDING)
        box_min_col = max(0, centroid_col - half_side - IMG_PADDING)
        box_max_col = min(mri_np.shape[2] - 1, centroid_col + half_side + IMG_PADDING)

        mri_crop = mri_np[slice_id, box_min_row:box_max_row+1, box_min_col:box_max_col+1]
        mask_crop = np_slice_mask[box_min_row:box_max_row+1, box_min_col:box_max_col+1]

        new_origin_x = origin[0] + (box_min_col * spacing[0]) 
        new_origin_y = origin[1] + (box_min_row * spacing[1]) 
        new_origin_z = origin[2] + (slice_id * spacing[2]) 

        slice_crops[slice_id] = {
            'image': mri_crop,
            'mask': mask_crop,
            'bbox': (box_min_row, box_max_row, box_min_col, box_max_col),
            'new_origin': (new_origin_x, new_origin_y, new_origin_z)
        }


    for i, (z1, z2, z3) in enumerate(triplets):
        crop1 = slice_crops[z1]['image']
        crop2 = slice_crops[z2]['image']
        crop3 = slice_crops[z3]['image']
        
        mask1 = slice_crops[z1]['mask']
        mask2 = slice_crops[z2]['mask']
        mask3 = slice_crops[z3]['mask']

        new_origin = slice_crops[z1]['new_origin']

        max_height = max(crop1.shape[0], crop2.shape[0], crop3.shape[0])
        max_width = max(crop1.shape[1], crop2.shape[1], crop3.shape[1])

        crop1 = pad_to_size(crop1, max_height, max_width)
        crop2 = pad_to_size(crop2, max_height, max_width)
        crop3 = pad_to_size(crop3, max_height, max_width)
        
        mask1 = pad_to_size(mask1, max_height, max_width)
        mask2 = pad_to_size(mask2, max_height, max_width)
        mask3 = pad_to_size(mask3, max_height, max_width)

        image_stack = np.stack([crop1, crop2, crop3], axis=0)
        mask_stack = np.stack([mask1, mask2, mask3], axis=0)

        image_stack_sitk = sitk.GetImageFromArray(image_stack)
        image_stack_sitk.SetSpacing(spacing)
        image_stack_sitk.SetOrigin(new_origin)
        mask_stack_sitk = sitk.GetImageFromArray(mask_stack)
        mask_stack_sitk.SetSpacing(spacing)
        mask_stack_sitk.SetOrigin(new_origin)

        logger.info(f"generated sitk stack for {i+1}/{len(triplets)}, z123 is {(z1, z2, z3)}")

        list_image_stack_sitk.append(image_stack_sitk)
        list_mask_stack_sitk.append(mask_stack_sitk)
        
    
    return list_image_stack_sitk, list_mask_stack_sitk

In [15]:
if __name__ == "__main__":
    mri_folder = MRI_FOLDER
    annotation_folder = ANNOTATION_FOLDER

    output_dir = OUTPUT_DIR
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all files in both folders
    mri_files = [os.path.join(mri_folder, f) for f in os.listdir(mri_folder) 
                if f.endswith('.nii.gz')]
    
    annotation_files = [os.path.join(annotation_folder, f) for f in os.listdir(annotation_folder) 
                       if f.endswith('.nii.gz')]
    
    # Match MRI files with corresponding annotation files
    file_pairs = match_files(mri_files, annotation_files)

    logger.info(f"file pairs are {file_pairs}")

    logger.info(f"Found {len(file_pairs)} matching pairs out of {len(mri_files)} MRI files and {len(annotation_files)} annotation files")

    for mri_path, annotation_path in tqdm(file_pairs, desc="Processing file pairs", unit="pair"):
        logger.info(f"............Starting process for {mri_path} and {annotation_path}")
        subj_id = Path(mri_path).stem.split('.')[0]
        try:
            dataloader = DataLoader(mri_path, annotation_path)
            dataloader.load_data();
            node_labels = dataloader.node_labels
            logger.info(f"retrieved node_labels, which is {node_labels}")
            node_masks = dataloader.node_masks
            mri_np = dataloader.mri_np
            mri_image = dataloader.mri_image
            size = dataloader.size
            spacing = dataloader.spacing
            origin = dataloader.origin

            for label in node_labels:
                logger.info(f"processing node {label}")
                label_masks = node_masks[label]
                logger.info(f"retrieved label_masks, length is {len(label_masks)}")
                id_list = dataloader.get_slices_with_mask(label)
                logger.info(f"retrieved id_list, length is {len(id_list)}, this node appears in {id_list}")
            
                # TODO: Get 2dx3
                list_image_stack_sitk, list_mask_stack_sitk = get2dx3(label, label_masks, id_list, mri_np, spacing, origin)
                
                if list_image_stack_sitk:
                    i = 0
                    for i in range(len(list_image_stack_sitk)):
                        output_filename = f"{os.path.basename(subj_id)}_node{label}_2dx3_{i}.nii.gz"
                        output_path = os.path.join(output_dir, output_filename)

                        sitk.WriteImage(list_image_stack_sitk[i], output_path)

                        logger.info(f"saved file with filename {output_filename}")
                        logger.info(f"it has size: {list_image_stack_sitk[i].GetSize()} and spacing {list_image_stack_sitk[i].GetSpacing()}")
                else:
                    logger.info(f"no images can be extracted")

        except Exception as e:
            logger.error(f"Error processing {mri_path}: {str(e)}")
            continue

2025-07-18 14:01:23,952 - INFO - file pairs are [('data/raw/images/1058-T2_FS_TRA+301.nii.gz', 'data/raw/labels/1058-T2_FS_TRA+301.nii.gz'), ('data/raw/images/985-T2_FS_TRA+301.nii.gz', 'data/raw/labels/985-T2_FS_TRA+301.nii.gz'), ('data/raw/images/856-NPC_T2W_SPIR_TRA+401.nii.gz', 'data/raw/labels/856-NPC_T2W_SPIR_TRA+401.nii.gz'), ('data/raw/images/1041-T2_FS_TRA+401.nii.gz', 'data/raw/labels/1041-T2_FS_TRA+401.nii.gz'), ('data/raw/images/926-T2_FS_TRA+301.nii.gz', 'data/raw/labels/926-T2_FS_TRA+301.nii.gz'), ('data/raw/images/1067-T2_FS_TRA+301.nii.gz', 'data/raw/labels/1067-T2_FS_TRA+301.nii.gz'), ('data/raw/images/860-T2_FS_TRA+301.nii.gz', 'data/raw/labels/860-T2_FS_TRA+301.nii.gz'), ('data/raw/images/1146-T2_FS_TRA+301.nii.gz', 'data/raw/labels/1146-T2_FS_TRA+301.nii.gz'), ('data/raw/images/1064-T2_FS_TRA+301.nii.gz', 'data/raw/labels/1064-T2_FS_TRA+301.nii.gz'), ('data/raw/images/1073-T2_FS_TRA.+701.nii.gz', 'data/raw/labels/1073-T2_FS_TRA.+701.nii.gz'), ('data/raw/images/859-T

Processing file pairs:   0%|          | 0/172 [00:00<?, ?pair/s]

2025-07-18 14:01:23,956 - INFO - ............Starting process for data/raw/images/1058-T2_FS_TRA+301.nii.gz and data/raw/labels/1058-T2_FS_TRA+301.nii.gz
2025-07-18 14:01:23,957 - INFO - DataLoader initialized
2025-07-18 14:01:23,958 - INFO - Loading MRI image from data/raw/images/1058-T2_FS_TRA+301.nii.gz


2025-07-18 14:01:24,298 - INFO - Loading annotation image from data/raw/labels/1058-T2_FS_TRA+301.nii.gz
2025-07-18 14:01:24,334 - INFO - Size match: True, Spacing match: True, Origin match: True
2025-07-18 14:01:24,335 - INFO - MRI spacing: (0.44920000433921814, 0.44920000433921814, 4.0), Anno spacing: (0.44920000433921814, 0.44920000433921814, 4.0)
2025-07-18 14:01:24,336 - INFO - xyz: (512, 512, 30), num_slides: 30
2025-07-18 14:01:24,337 - INFO - Image spacing: (0.44920000433921814, 0.44920000433921814, 4.0)
2025-07-18 14:01:24,338 - INFO - Image origin: (-109.07538604736328, -160.77491760253906, -22.91573715209961)
2025-07-18 14:01:24,338 - INFO - Image size: (512, 512, 30)
2025-07-18 14:01:24,441 - INFO - Found 4 lymph node annotations with labels: [1 2 3 4]
2025-07-18 14:01:24,442 - INFO - Creating mask for node 1
2025-07-18 14:01:24,488 - INFO -   Node 1 stats: {'mean_intensity': np.float64(50.67224785248524), 'std_intensity': np.float64(9.259288004160709), 'volume_mm3': np.flo

Processing file pairs:   1%|          | 1/172 [00:00<01:58,  1.44pair/s]

...

Processing file pairs:  99%|█████████▉| 170/172 [01:53<00:01,  1.49pair/s]

2025-07-18 14:03:17,147 - INFO - ............Starting process for data/raw/images/884-T2_FS_TRA+301.nii.gz and data/raw/labels/884-T2_FS_TRA+301.nii.gz
2025-07-18 14:03:17,148 - INFO - DataLoader initialized
2025-07-18 14:03:17,149 - INFO - Loading MRI image from data/raw/images/884-T2_FS_TRA+301.nii.gz
2025-07-18 14:03:17,467 - INFO - Loading annotation image from data/raw/labels/884-T2_FS_TRA+301.nii.gz
2025-07-18 14:03:17,503 - INFO - Size match: True, Spacing match: True, Origin match: True
2025-07-18 14:03:17,504 - INFO - MRI spacing: (0.44920000433921814, 0.44920000433921814, 4.0), Anno spacing: (0.44920000433921814, 0.44920000433921814, 4.0)
2025-07-18 14:03:17,505 - INFO - xyz: (512, 512, 30), num_slides: 30
2025-07-18 14:03:17,506 - INFO - Image spacing: (0.44920000433921814, 0.44920000433921814, 4.0)
2025-07-18 14:03:17,507 - INFO - Image origin: (-119.60105895996094, -153.80384826660156, -12.625197410583496)
2025-07-18 14:03:17,507 - INFO - Image size: (512, 512, 30)
2025-07

Processing file pairs:  99%|█████████▉| 171/172 [01:53<00:00,  1.52pair/s]

2025-07-18 14:03:17,773 - INFO - ............Starting process for data/raw/images/1095-T2_FS_TRA+301.nii.gz and data/raw/labels/1095-T2_FS_TRA+301.nii.gz
2025-07-18 14:03:17,773 - INFO - DataLoader initialized
2025-07-18 14:03:17,774 - INFO - Loading MRI image from data/raw/images/1095-T2_FS_TRA+301.nii.gz
2025-07-18 14:03:18,096 - INFO - Loading annotation image from data/raw/labels/1095-T2_FS_TRA+301.nii.gz
2025-07-18 14:03:18,133 - INFO - Size match: True, Spacing match: True, Origin match: True
2025-07-18 14:03:18,134 - INFO - MRI spacing: (0.44920000433921814, 0.44920000433921814, 4.0), Anno spacing: (0.44920000433921814, 0.44920000433921814, 4.0)
2025-07-18 14:03:18,135 - INFO - xyz: (512, 512, 30), num_slides: 30
2025-07-18 14:03:18,136 - INFO - Image spacing: (0.44920000433921814, 0.44920000433921814, 4.0)
2025-07-18 14:03:18,136 - INFO - Image origin: (-118.4688949584961, -149.03692626953125, -112.97413635253906)
2025-07-18 14:03:18,137 - INFO - Image size: (512, 512, 30)
2025

Processing file pairs: 100%|██████████| 172/172 [01:54<00:00,  1.50pair/s]
